# Mimi-VC Analysis: Full Diagnostic (Retrieval + Transformer)

1. **Cell 1** — Upload target speaker audio
2. **Cell 2** — Upload source audio
3. **Cells 3-4** — Install, load Mimi, core functions
4. **Cell 5** — Audio analysis (RMS, peak, duration)
5. **Cell 6** — Encode & token statistics
6. **Cell 7** — FAISS retrieval methods (Naive, Lookup, FAISS, FAISS+Smooth, keepCB0)
7. **Cell 7b** — Transformer V2 model definition
8. **Cell 7c** — Train Transformer V2 with per-CB loss/accuracy tracking
9. **Cell 7d** — Transformer inference (multiple temps, hybrid, teacher-forced, free-gen)
10. **Cell 7e** — Deep Transformer diagnostics (exposure bias, error cascade, distribution, confidence, self-recon, embedding distance)
11. **Cell 7f** — Transformer visual diagnostics (heatmaps, spectrograms, CB1 distribution, confidence heatmap)
12. **Cell 8** — Playback all methods
13. **Cells 9-10** — Retrieval token analysis + visual diagnostics
14. **Cell 11** — Objective metrics (SECS, MCD for all methods)
15. **Cell 12** — Save & summary
16. **Cell 13** — Download zip

In [ ]:
# ==============================================================
# CELL 1: Upload TARGET speaker audio (the voice to clone)
# ==============================================================
# You can upload multiple files — they will all be combined.
# Supported: WAV, MP3, FLAC, M4A, OGG

from google.colab import files
print('Upload TARGET speaker audio file(s):')
uploaded_target = files.upload()
print(f'\nReceived {len(uploaded_target)} file(s): {list(uploaded_target.keys())}')

In [ ]:
# ==============================================================
# CELL 2: Upload SOURCE audio (the speech to convert)
# ==============================================================

print('Upload SOURCE audio file(s):')
uploaded_source = files.upload()
print(f'\nReceived {len(uploaded_source)} file(s): {list(uploaded_source.keys())}')

In [ ]:
# ==============================================================
# CELL 3: Install & Setup
# ==============================================================
!pip install -q transformers torch torchaudio faiss-cpu librosa soundfile matplotlib resemblyzer pesq jiwer

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import faiss
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os, time, math, json
from collections import defaultdict, Counter
from transformers import MimiModel, AutoFeatureExtractor
from IPython.display import Audio as PlayAudio, display
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load Mimi
print('Loading Mimi codec...')
mimi = MimiModel.from_pretrained('kyutai/mimi').to(device).eval()
feat_ext = AutoFeatureExtractor.from_pretrained('kyutai/mimi')

SR = feat_ext.sampling_rate
N_CB = mimi.config.num_quantizers
CB_SIZE = mimi.config.codebook_size
CB_DIM = mimi.config.codebook_dim
FRAME_RATE = mimi.config.frame_rate

print(f'\n=== MIMI CODEC INFO ===')
print(f'Sample rate:   {SR} Hz')
print(f'Frame rate:    {FRAME_RATE} Hz ({1000/FRAME_RATE:.0f}ms per frame)')
print(f'Codebooks:     {N_CB} x {CB_SIZE} (dim={CB_DIM})')
print(f'Bitrate:       {N_CB * math.log2(CB_SIZE) * FRAME_RATE / 1000:.1f} kbps')
print(f'Quantizer:     {type(mimi.quantizer).__name__}')
if hasattr(mimi.quantizer, 'num_semantic_quantizers'):
    n_sem = mimi.quantizer.num_semantic_quantizers
    print(f'  Semantic CBs:  {n_sem} (CB0{"..CB"+str(n_sem-1) if n_sem>1 else ""})')
    print(f'  Acoustic CBs:  {N_CB - n_sem} (CB{n_sem}..CB{N_CB-1})')
print()

In [ ]:
# ==============================================================
# CELL 4: Core functions
# ==============================================================

def load_audio(path, target_sr=SR):
    wav, orig_sr = librosa.load(path, sr=None, mono=True)
    if orig_sr != target_sr:
        wav = librosa.resample(wav, orig_sr=orig_sr, target_sr=target_sr)
    return wav.astype(np.float32)

def encode(wav_np):
    inputs = feat_ext(raw_audio=wav_np, sampling_rate=SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mimi.encode(inputs['input_values'], inputs.get('padding_mask'))
    return out.audio_codes.cpu()

def decode(codes):
    with torch.no_grad():
        out = mimi.decode(codes.to(device))
    return out.audio_values.cpu().squeeze().numpy()

def get_codebook_embeddings(cb_idx=0):
    q = mimi.quantizer
    if hasattr(q, 'semantic_residual_vector_quantizer'):
        n_sem = q.num_semantic_quantizers
        if cb_idx < n_sem:
            layer = q.semantic_residual_vector_quantizer.layers[cb_idx]
        else:
            layer = q.acoustic_residual_vector_quantizer.layers[cb_idx - n_sem]
    elif hasattr(q, 'layers'):
        layer = q.layers[cb_idx]
    else:
        raise RuntimeError(f'Unknown quantizer: {type(q).__name__}')
    for attr in ['codebook.embed', 'codebook.embed_sum', 'codebook.embedding.weight']:
        obj = layer
        try:
            for part in attr.split('.'):
                obj = getattr(obj, part)
            w = obj.detach().cpu().float()
            if w.dim() == 3: w = w.squeeze(0)
            if w.shape[0] >= CB_SIZE: return w
        except AttributeError:
            continue
    raise RuntimeError('Cannot find codebook embeddings')

cb0_emb = get_codebook_embeddings(0).numpy()
print(f'CB0 embeddings: {cb0_emb.shape}')
print('Functions ready.')

In [ ]:
# ==============================================================
# CELL 5: Load & analyze uploaded audio
# ==============================================================

CHUNK_SEC = 30
chunk_samples = int(CHUNK_SEC * SR)

# --- Load target ---
print('='*70)
print('TARGET SPEAKER AUDIO ANALYSIS')
print('='*70)

target_wavs = []
for fname, data in uploaded_target.items():
    tmp = f'/tmp/target_{fname}'
    with open(tmp, 'wb') as f:
        f.write(data)
    wav = load_audio(tmp)
    dur = len(wav) / SR
    rms = np.sqrt(np.mean(wav**2))
    peak = np.max(np.abs(wav))
    print(f'  File:     {fname}')
    print(f'  Duration: {dur:.2f}s ({int(dur*FRAME_RATE)} frames)')
    print(f'  RMS:      {rms:.4f}  Peak: {peak:.4f}  SNR est: {20*np.log10(peak/max(rms*0.01, 1e-10)):.1f} dB')
    target_wavs.append(wav)

total_target_dur = sum(len(w) for w in target_wavs) / SR
target_wav_concat = np.concatenate(target_wavs)
print(f'  TOTAL:    {total_target_dur:.2f}s from {len(target_wavs)} file(s)')

print(f'\nPlayback (first 15s):')
display(PlayAudio(target_wav_concat[:int(SR*15)], rate=SR))

# --- Load source ---
print('\n' + '='*70)
print('SOURCE AUDIO ANALYSIS')
print('='*70)

source_wavs = []
for fname, data in uploaded_source.items():
    tmp = f'/tmp/source_{fname}'
    with open(tmp, 'wb') as f:
        f.write(data)
    wav = load_audio(tmp)
    dur = len(wav) / SR
    rms = np.sqrt(np.mean(wav**2))
    peak = np.max(np.abs(wav))
    print(f'  File:     {fname}')
    print(f'  Duration: {dur:.2f}s ({int(dur*FRAME_RATE)} frames)')
    print(f'  RMS:      {rms:.4f}  Peak: {peak:.4f}')
    source_wavs.append(wav)

source_wav = np.concatenate(source_wavs)
source_dur = len(source_wav) / SR
print(f'  TOTAL:    {source_dur:.2f}s')

print(f'\nPlayback:')
display(PlayAudio(source_wav, rate=SR))

In [ ]:
# ==============================================================
# CELL 6: Encode everything & analyze token statistics
# ==============================================================

print('='*70)
print('ENCODING & TOKEN ANALYSIS')
print('='*70)

# Encode target (in chunks)
print('\nEncoding target speaker...')
tgt_chunks = []
for wav in target_wavs:
    for i in range(0, len(wav), chunk_samples):
        chunk = wav[i:i+chunk_samples]
        if len(chunk) > SR:
            tgt_chunks.append(chunk)

tgt_codes_list = []
for i, chunk in enumerate(tgt_chunks):
    codes = encode(chunk)
    tgt_codes_list.append(codes)
    print(f'  Chunk {i+1}/{len(tgt_chunks)}: {codes.shape[2]} frames')

all_tgt_codes = torch.cat(tgt_codes_list, dim=2)  # [1, 8, T_tgt]
T_tgt = all_tgt_codes.shape[2]

# Encode source
print('\nEncoding source...')
src_codes = encode(source_wav)
T_src = src_codes.shape[2]
print(f'  Source: {T_src} frames ({T_src/FRAME_RATE:.1f}s)')

# Reconstruction test
print('\n--- Reconstruction test (encode -> decode) ---')
src_recon = decode(src_codes)
tgt_recon = decode(all_tgt_codes[:, :, :min(T_tgt, 500)])
print(f'  Source recon length: {len(src_recon)/SR:.2f}s (original: {source_dur:.2f}s)')

# Token statistics per codebook
print('\n--- Per-codebook token statistics ---')
print(f'{"CB":>4} {"Role":>10} {"Src Unique":>12} {"Tgt Unique":>12} {"Overlap":>10} {"Coverage%":>10}')
print('-'*65)

for cb in range(N_CB):
    s_tokens = set(src_codes[0, cb].numpy().tolist())
    t_tokens = set(all_tgt_codes[0, cb].numpy().tolist())
    overlap = s_tokens & t_tokens
    coverage = len(overlap) / len(s_tokens) * 100 if s_tokens else 0
    role = 'semantic' if cb == 0 else 'acoustic'
    print(f'{cb:>4} {role:>10} {len(s_tokens):>12} {len(t_tokens):>12} {len(overlap):>10} {coverage:>9.1f}%')

# CB0 detailed analysis
print('\n--- CB0 (semantic) detailed analysis ---')
src_cb0 = src_codes[0, 0].numpy()
tgt_cb0 = all_tgt_codes[0, 0].numpy()

src_cb0_counter = Counter(src_cb0.tolist())
tgt_cb0_counter = Counter(tgt_cb0.tolist())

# Find source tokens NOT in target (these will have bad retrieval)
missing = set(src_cb0_counter.keys()) - set(tgt_cb0_counter.keys())
missing_frames = sum(src_cb0_counter[t] for t in missing)
print(f'  Source CB0 unique tokens: {len(src_cb0_counter)}')
print(f'  Target CB0 unique tokens: {len(tgt_cb0_counter)}')
print(f'  Source tokens missing in target: {len(missing)} ({missing_frames}/{T_src} frames = {100*missing_frames/T_src:.1f}%)')

if missing:
    print(f'  Missing token IDs: {sorted(list(missing))[:20]}' + (' ...' if len(missing) > 20 else ''))

# CB0 embedding similarity between source and target
src_emb = cb0_emb[src_cb0].astype(np.float32)
tgt_emb_all = cb0_emb[tgt_cb0].astype(np.float32)
src_emb_n = src_emb / (np.linalg.norm(src_emb, axis=1, keepdims=True) + 1e-8)
tgt_emb_n = tgt_emb_all / (np.linalg.norm(tgt_emb_all, axis=1, keepdims=True) + 1e-8)
# Average cosine similarity of each source frame to its nearest target frame
faiss.normalize_L2(src_emb)
faiss.normalize_L2(tgt_emb_all)
tmp_idx = faiss.IndexFlatIP(cb0_emb.shape[1])
tmp_idx.add(tgt_emb_all)
dists, _ = tmp_idx.search(src_emb, 1)
print(f'  Avg cosine sim (src->nearest tgt): {dists.mean():.4f} (min={dists.min():.4f}, max={dists.max():.4f})')

# Token entropy
for label, counter, T in [('Source', src_cb0_counter, T_src), ('Target', tgt_cb0_counter, T_tgt)]:
    probs = np.array(list(counter.values())) / T
    entropy = -np.sum(probs * np.log2(probs + 1e-10))
    print(f'  {label} CB0 entropy: {entropy:.2f} bits (max={math.log2(CB_SIZE):.2f})')

In [ ]:
# ==============================================================
# CELL 7: Build FAISS index & run all conversion methods
# ==============================================================

print('='*70)
print('VOICE CONVERSION — ALL METHODS')
print('='*70)

# Build index
all_sem = []
all_full = []
all_acou = []
for t in range(T_tgt):
    all_sem.append(all_tgt_codes[0, 0, t].item())
    all_full.append(all_tgt_codes[0, :, t].numpy())
    all_acou.append(all_tgt_codes[0, 1:, t].numpy())

sem_arr = np.array(all_sem)
full_arr = np.stack(all_full)
acou_arr = np.stack(all_acou)

# FAISS
tgt_emb = cb0_emb[sem_arr].astype(np.float32)
faiss.normalize_L2(tgt_emb)
faiss_index = faiss.IndexFlatIP(cb0_emb.shape[1])
faiss_index.add(tgt_emb)

# Lookup table
lookup = defaultdict(list)
for i, s in enumerate(all_sem):
    lookup[s].append(i)

print(f'Index: {faiss_index.ntotal} frames, {len(lookup)} unique CB0 tokens')

# --- Method 1: Naive swap ---
print('\n--- Method 1: Naive Swap ---')
min_len = min(T_src, T_tgt)
naive_codes = src_codes[:, :, :min_len].clone()
naive_codes[:, 1:, :] = all_tgt_codes[:, 1:, :min_len]
naive_wav = decode(naive_codes)
print(f'  Swapped CB1-7 from target (first {min_len} frames)')

# --- Method 2: Retrieval lookup ---
print('\n--- Method 2: Retrieval (Lookup) ---')
lookup_codes = src_codes.clone()
n_matched = 0
for t in range(T_src):
    sem = src_codes[0, 0, t].item()
    if sem in lookup:
        idx = np.random.choice(lookup[sem])
        lookup_codes[0, :, t] = torch.tensor(full_arr[idx])
        n_matched += 1
lookup_wav = decode(lookup_codes)
print(f'  Exact match: {n_matched}/{T_src} frames ({100*n_matched/T_src:.1f}%)')

# --- Method 3: FAISS ---
print('\n--- Method 3: Retrieval (FAISS) ---')
src_sem = src_codes[0, 0].numpy()
src_emb_q = cb0_emb[src_sem].astype(np.float32)
faiss.normalize_L2(src_emb_q)
K = 8
dists_k, indices_k = faiss_index.search(src_emb_q, K)

faiss_codes = src_codes.clone()
for t in range(T_src):
    weights = F.softmax(torch.tensor(dists_k[t]) * 10, dim=0).numpy()
    chosen = np.random.choice(K, p=weights)
    faiss_codes[0, :, t] = torch.tensor(full_arr[indices_k[t, chosen]])
faiss_wav = decode(faiss_codes)
print(f'  k={K}, avg top-1 cosine: {dists_k[:, 0].mean():.4f}')

# --- Method 4: FAISS + Smooth ---
print('\n--- Method 4: Retrieval (FAISS+Smooth) ---')
ALPHA = 0.5
smooth_codes = src_codes.clone()
selected = np.zeros(T_src, dtype=np.int64)
selected[0] = indices_k[0, 0]
for t in range(1, T_src):
    cands = indices_k[t]
    sem_s = dists_k[t]
    tmp_s = 1.0 / (1.0 + np.abs(cands - selected[t-1]).astype(np.float32))
    combined = ALPHA * sem_s + (1-ALPHA) * tmp_s
    selected[t] = cands[np.argmax(combined)]

for t in range(T_src):
    smooth_codes[0, :, t] = torch.tensor(full_arr[selected[t]])
smooth_wav = decode(smooth_codes)

# Temporal smoothness: measure how many frame jumps in target index
jumps = np.abs(np.diff(selected))
print(f'  alpha={ALPHA}, k={K}')
print(f'  Temporal jumps: mean={jumps.mean():.1f}, median={np.median(jumps):.1f}, max={jumps.max()}')
print(f'  Smooth transitions (jump<=5): {np.sum(jumps<=5)}/{len(jumps)} ({100*np.sum(jumps<=5)/len(jumps):.1f}%)')

# --- Method 5: FAISS+Smooth, keep source CB0 ---
print('\n--- Method 5: FAISS+Smooth (keep source CB0) ---')
smooth_keepcb0_codes = src_codes.clone()
for t in range(T_src):
    smooth_keepcb0_codes[0, 1:, t] = torch.tensor(full_arr[selected[t]][1:])
smooth_keepcb0_wav = decode(smooth_keepcb0_codes)
print(f'  Same as Method 4 but preserving source CB0 (content tokens)')

print('\nAll methods complete.')

In [ ]:
# ==============================================================
# CELL 7b: Transformer V2 — Model Definition
# ==============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_len=4096):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        t = torch.arange(max_len, dtype=inv_freq.dtype)
        freqs = torch.outer(t, inv_freq)
        self.register_buffer('cos_cached', freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer('sin_cached', freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, x):
        T = x.shape[2]
        return self.cos_cached[:, :, :T, :], self.sin_cached[:, :, :T, :]

def apply_rope(q, k, cos, sin):
    def rotate(x, c, s):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([x1*c - x2*s, x2*c + x1*s], dim=-1)
    return rotate(q, cos, sin), rotate(k, cos, sin)

class RoPETransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1, has_cross_attn=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.has_cross_attn = has_cross_attn
        self.norm1 = nn.LayerNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)
        self.rope = RotaryEmbedding(self.head_dim)
        if has_cross_attn:
            self.norm_cross = nn.LayerNorm(d_model)
            self.cq_proj = nn.Linear(d_model, d_model)
            self.ck_proj = nn.Linear(d_model, d_model)
            self.cv_proj = nn.Linear(d_model, d_model)
            self.co_proj = nn.Linear(d_model, d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(ff_dim, d_model), nn.Dropout(dropout))
        self.dropout = nn.Dropout(dropout)

    def _reshape(self, x, B, T):
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, cross_kv=None, causal=True):
        B, T, _ = x.shape
        h = self.norm1(x)
        q = self._reshape(self.q_proj(h), B, T)
        k = self._reshape(self.k_proj(h), B, T)
        v = self._reshape(self.v_proj(h), B, T)
        cos, sin = self.rope(q)
        q, k = apply_rope(q, k, cos, sin)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if causal:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            attn = attn.masked_fill(mask, float('-inf'))
        attn_w = F.softmax(attn, dim=-1)
        out = (attn_w @ v).transpose(1, 2).reshape(B, T, self.d_model)
        x = x + self.dropout(self.o_proj(out))
        if self.has_cross_attn and cross_kv is not None:
            h = self.norm_cross(x)
            T_kv = cross_kv.shape[1]
            cq = self._reshape(self.cq_proj(h), B, T)
            ck = self._reshape(self.ck_proj(cross_kv), B, T_kv)
            cv = self._reshape(self.cv_proj(cross_kv), B, T_kv)
            c_attn = F.softmax((cq @ ck.transpose(-2, -1)) / math.sqrt(self.head_dim), dim=-1)
            x = x + self.dropout(self.co_proj((c_attn @ cv).transpose(1, 2).reshape(B, T, self.d_model)))
        x = x + self.ffn(self.norm2(x))
        return x

class AcousticTranslatorV2(nn.Module):
    def __init__(self, n_vocab=2048, n_cb=7, d_model=256, n_heads=4,
                 n_enc_layers=4, n_src_acou_layers=2, n_dec_layers=4, dropout=0.1):
        super().__init__()
        self.n_cb = n_cb
        self.d_model = d_model
        ff_dim = d_model * 4
        self.sem_embed = nn.Embedding(n_vocab, d_model)
        self.acou_embed = nn.Embedding(n_vocab, d_model)
        self.cb_embed = nn.Embedding(n_cb + 1, d_model)
        self.src_acou_embeds = nn.ModuleList([nn.Embedding(n_vocab, d_model) for _ in range(n_cb)])
        self.sem_encoder = nn.ModuleList([RoPETransformerLayer(d_model, n_heads, ff_dim, dropout) for _ in range(n_enc_layers)])
        self.sem_norm = nn.LayerNorm(d_model)
        self.src_acou_encoder = nn.ModuleList([RoPETransformerLayer(d_model, n_heads, ff_dim, dropout) for _ in range(n_src_acou_layers)])
        self.src_acou_norm = nn.LayerNorm(d_model)
        self.decoder = nn.ModuleList([RoPETransformerLayer(d_model, n_heads, ff_dim, dropout, has_cross_attn=True) for _ in range(n_dec_layers)])
        self.dec_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, n_vocab)

    def encode_semantic(self, sem_tokens):
        x = self.sem_embed(sem_tokens)
        for layer in self.sem_encoder: x = layer(x, causal=True)
        return self.sem_norm(x)

    def encode_source_acoustic(self, src_acou_tokens):
        B, n_cb, T = src_acou_tokens.shape
        x = sum(self.src_acou_embeds[i](src_acou_tokens[:, i]) for i in range(n_cb)) / n_cb
        for layer in self.src_acou_encoder: x = layer(x, causal=False)
        return self.src_acou_norm(x)

    def forward(self, sem_tokens, src_acou_tokens, tgt_acou_tokens=None):
        B, T = sem_tokens.shape
        sem_enc = self.encode_semantic(sem_tokens)
        src_acou_enc = self.encode_source_acoustic(src_acou_tokens)
        all_logits = []
        prev_cb_emb = torch.zeros(B, T, self.d_model, device=sem_tokens.device)
        for cb_i in range(self.n_cb):
            cb_idx = torch.full((B, T), cb_i + 1, dtype=torch.long, device=sem_tokens.device)
            h = sem_enc + prev_cb_emb + self.cb_embed(cb_idx)
            for layer in self.decoder: h = layer(h, cross_kv=src_acou_enc, causal=True)
            logits = self.output_head(self.dec_norm(h))
            all_logits.append(logits)
            if tgt_acou_tokens is not None:
                prev_cb_emb = self.acou_embed(tgt_acou_tokens[:, cb_i])
            else:
                prev_cb_emb = self.acou_embed(logits.argmax(dim=-1))
        return torch.stack(all_logits, dim=1)

    @torch.no_grad()
    def generate(self, sem_tokens, src_acou_tokens, temperature=0.8, top_k=50):
        B, T = sem_tokens.shape
        sem_enc = self.encode_semantic(sem_tokens)
        src_acou_enc = self.encode_source_acoustic(src_acou_tokens)
        predicted = []
        prev_cb_emb = torch.zeros(B, T, self.d_model, device=sem_tokens.device)
        for cb_i in range(self.n_cb):
            cb_idx = torch.full((B, T), cb_i + 1, dtype=torch.long, device=sem_tokens.device)
            h = sem_enc + prev_cb_emb + self.cb_embed(cb_idx)
            for layer in self.decoder: h = layer(h, cross_kv=src_acou_enc, causal=True)
            logits = self.output_head(self.dec_norm(h)) / temperature
            if top_k > 0:
                topk_v, _ = torch.topk(logits, top_k, dim=-1)
                logits[logits < topk_v[:, :, -1:]] = float('-inf')
            tokens = torch.multinomial(F.softmax(logits, dim=-1).reshape(-1, n_vocab), 1).reshape(B, T)
            predicted.append(tokens)
            prev_cb_emb = self.acou_embed(tokens)
        return torch.stack(predicted, dim=1)

n_vocab = CB_SIZE
print('AcousticTranslatorV2 defined.')

In [ ]:
# ==============================================================
# CELL 7c: Train Transformer V2 with per-CB loss tracking
# ==============================================================

D_MODEL = 256; N_HEADS = 4; SEQ_LEN = 64; BATCH_SIZE = 8; LR = 3e-4
N_STEPS = min(int(500 * max(1, T_tgt / 500)), 3000)
N_ACOU = N_CB - 1  # number of acoustic codebooks

cb_weights = torch.tensor([2.0**(-(i)/2) for i in range(N_ACOU)], device=device)
cb_weights = cb_weights / cb_weights.sum() * N_ACOU

print(f'=== TRANSFORMER V2 TRAINING ===')
print(f'Data: {T_tgt} frames ({T_tgt/FRAME_RATE:.1f}s)')
print(f'Steps: {N_STEPS}, Batch: {BATCH_SIZE}, Seq: {SEQ_LEN}')
print(f'Acoustic codebooks: {N_ACOU} (CB1..CB{N_CB-1})')
print(f'Loss weights (first 8): {[f"CB{i+1}={w:.2f}" for i, w in enumerate(cb_weights[:8].cpu())]}')

train_sem = all_tgt_codes[0, 0]
train_acou = all_tgt_codes[0, 1:]

model_v2 = AcousticTranslatorV2(
    n_vocab=CB_SIZE, n_cb=N_ACOU, d_model=D_MODEL, n_heads=N_HEADS,
    n_enc_layers=4, n_src_acou_layers=2, n_dec_layers=4
).to(device)
n_params = sum(p.numel() for p in model_v2.parameters())
print(f'Params: {n_params/1e6:.1f}M\n')

opt = torch.optim.AdamW(model_v2.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, N_STEPS)

# Track per-CB losses for diagnostics
all_losses = []
per_cb_losses = {i: [] for i in range(N_ACOU)}
per_cb_accs = {i: [] for i in range(N_ACOU)}

model_v2.train()
t0 = time.time()
log_interval = max(1, N_STEPS // 20)

for step in range(N_STEPS):
    starts = torch.randint(0, max(1, T_tgt - SEQ_LEN), (BATCH_SIZE,))
    sem_b = torch.stack([train_sem[s:s+SEQ_LEN] for s in starts]).to(device)
    acou_b = torch.stack([train_acou[:, s:s+SEQ_LEN] for s in starts]).to(device)

    logits = model_v2(sem_b, acou_b, tgt_acou_tokens=acou_b)

    loss = 0
    for cb_i in range(N_ACOU):
        ce = F.cross_entropy(logits[:, cb_i].reshape(-1, CB_SIZE), acou_b[:, cb_i].reshape(-1))
        loss = loss + cb_weights[cb_i] * ce
        if (step + 1) % log_interval == 0:
            per_cb_losses[cb_i].append(ce.item())
            acc = (logits[:, cb_i].argmax(-1) == acou_b[:, cb_i]).float().mean().item()
            per_cb_accs[cb_i].append(acc * 100)
    loss = loss / N_ACOU

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)
    opt.step()
    sched.step()
    all_losses.append(loss.item())

    if (step + 1) % log_interval == 0:
        # Show first 8 CBs in log
        cb_acc_str = ' '.join([f'CB{i+1}:{per_cb_accs[i][-1]:.0f}%' for i in range(min(8, N_ACOU))])
        if N_ACOU > 8:
            cb_acc_str += f' ... CB{N_ACOU}:{per_cb_accs[N_ACOU-1][-1]:.0f}%'
        print(f'  Step {step+1:4d}/{N_STEPS} | Loss:{loss.item():.4f} | Acc: {cb_acc_str} | {time.time()-t0:.0f}s')

train_time = time.time() - t0
model_v2.eval()

# --- Plot training curves ---
# Show important CBs only (first 8 + last few) to keep plots readable
show_cbs = list(range(min(8, N_ACOU)))
if N_ACOU > 8:
    show_cbs += [N_ACOU - 1]  # also show last
n_show = len(show_cbs)
colors = plt.cm.tab10(np.linspace(0, 1, min(n_show, 10)))

fig, axes = plt.subplots(1, 3, figsize=(18, 4), constrained_layout=True)

axes[0].plot(all_losses, color='#7030A0', linewidth=0.5)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Weighted Loss')
axes[0].set_title(f'Total Loss ({N_STEPS} steps, {train_time:.0f}s)')
axes[0].grid(True, alpha=0.3)

for j, i in enumerate(show_cbs):
    axes[1].plot(per_cb_losses[i], label=f'CB{i+1}', color=colors[j % len(colors)])
axes[1].set_xlabel(f'Log point (every {log_interval} steps)'); axes[1].set_ylabel('CE Loss')
axes[1].set_title(f'Per-Codebook Loss (showing {n_show}/{N_ACOU})')
axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)

for j, i in enumerate(show_cbs):
    axes[2].plot(per_cb_accs[i], label=f'CB{i+1}', color=colors[j % len(colors)])
axes[2].set_xlabel(f'Log point'); axes[2].set_ylabel('Accuracy %')
axes[2].set_title('Per-Codebook Token Accuracy (teacher forcing)')
axes[2].legend(fontsize=7); axes[2].grid(True, alpha=0.3)

plt.savefig('/content/transformer_training.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Full accuracy summary ---
print(f'\n=== TRAINING SUMMARY ===')
print(f'Final loss: {all_losses[-1]:.4f}')
print(f'{"CB":<6} {"Loss":>8} {"Acc%":>8} {"Status":>20}')
print('-'*45)
for i in range(N_ACOU):
    acc = per_cb_accs[i][-1]
    loss_val = per_cb_losses[i][-1]
    if acc > 90: status = 'GOOD'
    elif acc > 50: status = 'Moderate'
    elif acc > 10: status = 'Weak'
    else: status = 'NOT LEARNING'
    print(f'CB{i+1:<4} {loss_val:>8.4f} {acc:>7.1f}% {status:>20}')

# Identify the "learnable boundary"
good_cbs = [i for i in range(N_ACOU) if per_cb_accs[i][-1] > 50]
weak_cbs = [i for i in range(N_ACOU) if 10 < per_cb_accs[i][-1] <= 50]
dead_cbs = [i for i in range(N_ACOU) if per_cb_accs[i][-1] <= 10]
print(f'\nLearnable (>50%): CB{[i+1 for i in good_cbs]} ({len(good_cbs)} CBs)')
print(f'Weak (10-50%):    CB{[i+1 for i in weak_cbs]} ({len(weak_cbs)} CBs)')
print(f'Dead (<10%):      CB{[i+1 for i in dead_cbs]} ({len(dead_cbs)} CBs)')
print(f'\nRECOMMENDATION: Use Transformer for CB1-{good_cbs[-1]+1 if good_cbs else 0}, retrieval for rest.')

In [ ]:
# ==============================================================
# CELL 7d: Transformer V2 Inference — all variants
# ==============================================================

print('='*70)
print('TRANSFORMER V2 — INFERENCE ON SOURCE AUDIO')
print('='*70)

src_sem_d = src_codes[0, 0:1].to(device)        # [1, T_src]
src_acou_d = src_codes[0, 1:].unsqueeze(0).to(device)  # [1, N_ACOU, T_src]

# Determine hybrid split based on training accuracy
hybrid_boundary = len(good_cbs) if good_cbs else 4  # how many CBs to use from transformer
print(f'Hybrid split: Transformer for CB1-{hybrid_boundary}, retrieval for CB{hybrid_boundary+1}-{N_CB-1}')

# --- V2 full (all temps) ---
v2_results = {}
for temp in [0.2, 0.5, 0.8, 1.0]:
    torch.manual_seed(42)
    pred = model_v2.generate(src_sem_d, src_acou_d, temperature=temp, top_k=50)
    vc = src_codes.clone()
    vc[0, 1:] = pred[0].cpu()
    w = decode(vc)
    v2_results[temp] = (vc, w, pred[0].cpu())
    print(f'  V2 temp={temp}: generated {T_src} frames')

# --- Hybrid (V2 for learnable CBs, retrieval for rest) ---
print(f'\n--- Hybrid (V2 for CB1-{hybrid_boundary}, FAISS+Smooth for CB{hybrid_boundary+1}-{N_CB-1}) ---')
hybrid_codes = src_codes.clone()
hybrid_codes[0, 1:1+hybrid_boundary] = v2_results[0.8][2][:hybrid_boundary]
hybrid_codes[0, 1+hybrid_boundary:] = smooth_codes[0, 1+hybrid_boundary:]
hybrid_wav = decode(hybrid_codes)

# --- Teacher-forced output (oracle — what model predicts with GT previous CBs) ---
print('\n--- Teacher-forced inference on target data (self-reconstruction test) ---')
eval_len = min(T_tgt, 200)
with torch.no_grad():
    tf_sem = all_tgt_codes[0, 0:1, :eval_len].to(device)
    tf_acou = all_tgt_codes[0, 1:, :eval_len].unsqueeze(0).to(device)
    tf_logits = model_v2(tf_sem, tf_acou, tgt_acou_tokens=tf_acou)

tf_codes = all_tgt_codes[:, :, :eval_len].clone()
tf_pred_tokens = tf_logits.argmax(-1)[0].cpu()  # [N_ACOU, eval_len]
tf_codes[0, 1:] = tf_pred_tokens
tf_wav = decode(tf_codes)

# Free-generation on same target data
torch.manual_seed(42)
fg_pred = model_v2.generate(tf_sem, tf_acou, temperature=0.8, top_k=50)
fg_codes = all_tgt_codes[:, :, :eval_len].clone()
fg_codes[0, 1:] = fg_pred[0].cpu()
fg_wav = decode(fg_codes)

print(f'  Teacher-forced on {eval_len} target frames')
print(f'  Free-generated on same {eval_len} target frames')

# --- Playback ---
print(f'\n{"="*70}')
print('TRANSFORMER RESULTS — Listen')
print(f'{"="*70}')

print('\n0. Source (original):')
display(PlayAudio(source_wav, rate=SR))

for temp in [0.2, 0.5, 0.8, 1.0]:
    print(f'\nV2 temp={temp}:')
    display(PlayAudio(v2_results[temp][1], rate=SR))

print(f'\nHybrid (V2 CB1-{hybrid_boundary} + Retrieval CB{hybrid_boundary+1}+):')
display(PlayAudio(hybrid_wav, rate=SR))

print(f'\nFAISS+Smooth (baseline):')
display(PlayAudio(smooth_wav, rate=SR))

print(f'\nTeacher-forced (target self-recon, argmax):')
display(PlayAudio(tf_wav, rate=SR))

print(f'\nFree-gen on target data (exposure bias test):')
display(PlayAudio(fg_wav, rate=SR))

print(f'\nTarget reference:')
display(PlayAudio(target_wav_concat[:int(SR*15)], rate=SR))

In [ ]:
# ==============================================================
# CELL 7e: DEEP TRANSFORMER DIAGNOSTICS
# ==============================================================

print('='*70)
print('DEEP TRANSFORMER V2 DIAGNOSTICS')
print('='*70)

# Focus analysis on first N_DIAG codebooks (the ones that matter most)
N_DIAG = min(8, N_ACOU)  # detailed analysis for first 8 acoustic CBs

# ---- 1. Teacher Forcing vs Free Generation Gap (Exposure Bias) ----
print(f'\n--- 1. EXPOSURE BIAS: Teacher-Forced vs Free-Gen Accuracy ---')
print(f'  (Both evaluated on TARGET data — measures error cascade)')
gt_acou = all_tgt_codes[0, 1:, :eval_len].numpy()

print(f'{"CB":<6} {"TF Acc%":>10} {"FreeGen Acc%":>12} {"Gap":>8} {"Diagnosis":>30}')
print('-'*70)
for cb_i in range(N_ACOU):
    tf_acc = np.mean(tf_pred_tokens[cb_i].numpy() == gt_acou[cb_i]) * 100
    fg_acc = np.mean(fg_pred[0, cb_i].cpu().numpy() == gt_acou[cb_i]) * 100
    gap = tf_acc - fg_acc
    if tf_acc < 5:
        diag = 'NOT LEARNING'
    elif gap > 20:
        diag = 'SEVERE exposure bias'
    elif gap > 10:
        diag = 'Moderate exposure bias'
    elif tf_acc < 15:
        diag = 'Weak learning'
    else:
        diag = 'OK'
    print(f'CB{cb_i+1:<4} {tf_acc:>10.1f} {fg_acc:>12.1f} {gap:>7.1f}  {diag:>30}')

# ---- 2. Error Cascade Analysis (first N_DIAG CBs) ----
print(f'\n--- 2. ERROR CASCADE: How CB(i) errors affect CB(i+1) ---')
fg_tokens = fg_pred[0].cpu().numpy()  # [N_ACOU, eval_len]
for cb_i in range(min(N_DIAG, N_ACOU - 1)):
    this_wrong = fg_tokens[cb_i] != gt_acou[cb_i]
    next_wrong = fg_tokens[cb_i+1] != gt_acou[cb_i+1]
    both_wrong = this_wrong & next_wrong
    if this_wrong.sum() > 0:
        cascade_rate = both_wrong.sum() / this_wrong.sum() * 100
    else:
        cascade_rate = 0
    print(f'  CB{cb_i+1} wrong -> CB{cb_i+2} also wrong: {cascade_rate:.1f}% ({both_wrong.sum()}/{this_wrong.sum()} frames)')

# ---- 3. Token Distribution Analysis (first N_DIAG CBs) ----
print(f'\n--- 3. TOKEN DISTRIBUTION: Predicted vs Ground Truth ---')
v2_pred = v2_results[0.8][2].numpy()  # [N_ACOU, T_src]

def token_entropy(arr):
    c = Counter(arr.tolist())
    p = np.array(list(c.values())) / len(arr)
    return -np.sum(p * np.log2(p + 1e-10))

for cb_i in range(N_DIAG):
    gt_unique = len(set(gt_acou[cb_i].tolist()))
    tf_unique = len(set(tf_pred_tokens[cb_i].numpy().tolist()))
    fg_unique = len(set(fg_tokens[cb_i].tolist()))
    v2_unique = len(set(v2_pred[cb_i].tolist()))
    tgt_full_unique = len(set(all_tgt_codes[0, cb_i+1].numpy().tolist()))

    gt_ent = token_entropy(gt_acou[cb_i])
    tf_ent = token_entropy(tf_pred_tokens[cb_i].numpy())
    fg_ent = token_entropy(fg_tokens[cb_i])
    v2_ent = token_entropy(v2_pred[cb_i])

    print(f'\n  CB{cb_i+1}:')
    print(f'    {"":15} {"Unique":>8} {"Entropy":>8}')
    print(f'    {"GT (target)":15} {gt_unique:>8} {gt_ent:>8.2f}')
    print(f'    {"TF argmax":15} {tf_unique:>8} {tf_ent:>8.2f}')
    print(f'    {"Free-gen":15} {fg_unique:>8} {fg_ent:>8.2f}')
    print(f'    {"V2 on source":15} {v2_unique:>8} {v2_ent:>8.2f}')
    print(f'    {"Full target":15} {tgt_full_unique:>8} (all {T_tgt} frames)')

    if fg_ent < gt_ent * 0.5:
        print(f'    >> WARNING: Free-gen entropy much lower than GT — model is mode-collapsing!')
    if fg_unique < gt_unique * 0.3:
        print(f'    >> WARNING: Free-gen uses very few unique tokens — repetitive output!')

# ---- 4. Per-frame Confidence Analysis ----
print(f'\n--- 4. PREDICTION CONFIDENCE ---')
with torch.no_grad():
    src_logits = model_v2(src_sem_d, src_acou_d, tgt_acou_tokens=None)
for cb_i in range(N_DIAG):
    probs = F.softmax(src_logits[0, cb_i].cpu(), dim=-1)
    top1_conf = probs.max(dim=-1).values.numpy()
    top5_sum = probs.topk(5, dim=-1).values.sum(dim=-1).numpy()
    print(f'  CB{cb_i+1}: top-1 conf mean={top1_conf.mean():.4f} std={top1_conf.std():.4f} | '
          f'top-5 mass={top5_sum.mean():.4f}')
    if top1_conf.mean() > 0.8:
        print(f'       >> Very high confidence — may be overfitting or mode collapse')
    elif top1_conf.mean() < 0.05:
        print(f'       >> Very low confidence — model is uncertain, near random')

# ---- 5. Self-Reconstruction Test ----
print(f'\n--- 5. SAME-SPEAKER SELF-RECONSTRUCTION TEST ---')
tgt_recon_codes = all_tgt_codes[:, :, :eval_len].clone()
torch.manual_seed(42)
tgt_self_pred = model_v2.generate(tf_sem, tf_acou, temperature=0.5, top_k=50)
tgt_recon_codes[0, 1:] = tgt_self_pred[0].cpu()
tgt_recon_wav = decode(tgt_recon_codes)
tgt_orig_wav = decode(all_tgt_codes[:, :, :eval_len])

from resemblyzer import VoiceEncoder, preprocess_wav
try: spk_enc
except: spk_enc = VoiceEncoder(device=device)

def compute_secs(a, b):
    a16 = librosa.resample(a.astype(np.float32), orig_sr=SR, target_sr=16000)
    b16 = librosa.resample(b.astype(np.float32), orig_sr=SR, target_sr=16000)
    ea = spk_enc.embed_utterance(a16)
    eb = spk_enc.embed_utterance(b16)
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

self_secs = compute_secs(tgt_recon_wav, tgt_orig_wav)
print(f'  Target self-reconstruction SECS: {self_secs:.4f}')
if self_secs < 0.7:
    print(f'  >> CRITICAL: Model cannot even reconstruct target speaker! ({self_secs:.4f} < 0.7)')
    print(f'     This means the model has NOT learned the target speaker\'s voice.')
elif self_secs < 0.85:
    print(f'  >> Model partially learned target voice, but not well enough.')
else:
    print(f'  >> Good self-reconstruction. Issue may be cross-speaker generalization.')

# ---- 6. Codebook Embedding Distance Analysis ----
print(f'\n--- 6. PREDICTED TOKEN EMBEDDING DISTANCES ---')
print(f'  (Are predicted tokens close to GT in embedding space, even if not exact match?)')
for cb_i in range(N_DIAG):
    cb_emb_i = get_codebook_embeddings(cb_i + 1).numpy()
    gt_tok = gt_acou[cb_i]
    fg_tok = fg_tokens[cb_i]
    gt_vecs = cb_emb_i[gt_tok]
    fg_vecs = cb_emb_i[fg_tok]
    gt_n = gt_vecs / (np.linalg.norm(gt_vecs, axis=1, keepdims=True) + 1e-8)
    fg_n = fg_vecs / (np.linalg.norm(fg_vecs, axis=1, keepdims=True) + 1e-8)
    cos_sim = np.sum(gt_n * fg_n, axis=1)
    l2_dist = np.linalg.norm(gt_vecs - fg_vecs, axis=1)
    print(f'  CB{cb_i+1}: cosine_sim mean={cos_sim.mean():.4f} std={cos_sim.std():.4f} | '
          f'L2 mean={l2_dist.mean():.4f}')
    if cos_sim.mean() > 0.8:
        print(f'       >> Embeddings are close even if tokens differ — soft accuracy is decent')
    elif cos_sim.mean() < 0.3:
        print(f'       >> Embeddings are far apart — model predictions are essentially random')

print(f'\n{"="*70}')
print('TRANSFORMER DIAGNOSIS COMPLETE')
print(f'{"="*70}')
print(f'''
INTERPRETATION GUIDE:
- If TF acc is high but FreeGen acc is low: EXPOSURE BIAS
  -> Fix: scheduled sampling, or switch to non-AR generation
- If both TF and FreeGen acc are low: MODEL NOT LEARNING
  -> Fix: more training data, larger model, or lower learning rate
- If cascade rate > 80%: ERRORS COMPOUND across codebooks
  -> Fix: use hybrid mode (transformer for learnable CBs only, retrieval for rest)
- If entropy is very low: MODE COLLAPSE
  -> Fix: higher temperature, reduce top-k, add dropout
- If self-recon SECS < 0.7: FUNDAMENTAL LEARNING FAILURE
  -> Fix: need much more target data (>5 min), or model too small
- If embedding distance is ok but token acc is low: SOFT ACCURACY OK
  -> The discrete bottleneck is the issue, not the model

Your model learned {len(good_cbs)} CBs well (>50%): CB{[i+1 for i in good_cbs]}
Recommendation: Transformer for CB1-{good_cbs[-1]+1 if good_cbs else 0}, retrieval for rest.
''')


In [ ]:
# ==============================================================
# CELL 7f: TRANSFORMER VISUAL DIAGNOSTICS
# ==============================================================

print('='*70)
print('TRANSFORMER VISUAL DIAGNOSTICS')
print('='*70)

L_v = min(T_src, eval_len, 200)

# --- 1. Token heatmaps: Source vs V2 vs Hybrid vs Retrieval vs Target ---
fig, axes = plt.subplots(7, 1, figsize=(16, 20), constrained_layout=True)
heat_items = [
    ('Source', src_codes[0, :, :L_v]),
    ('V2 temp=0.2', v2_results[0.2][0][0, :, :L_v]),
    ('V2 temp=0.8', v2_results[0.8][0][0, :, :L_v]),
    ('Hybrid (V2+Ret)', hybrid_codes[0, :, :L_v]),
    ('FAISS+Smooth', smooth_codes[0, :, :L_v]),
    ('Teacher-Forced (argmax)', tf_codes[0, :, :min(L_v, eval_len)]),
    ('Target Reference', all_tgt_codes[0, :, :L_v]),
]
for ax, (title, codes) in zip(axes, heat_items):
    im = ax.imshow(codes.numpy(), aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_ylabel('CB'); ax.set_title(title, fontsize=10)
    ax.set_yticks(range(0, N_CB, max(1, N_CB//8)))
    plt.colorbar(im, ax=ax, shrink=0.5)
axes[-1].set_xlabel('Frame')
plt.suptitle('Token Heatmaps — All Methods incl. Transformer', fontsize=13)
plt.savefig('/content/heatmaps_transformer.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 2. Per-CB change rate bar chart (first 10 CBs for readability) ---
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
n_show_bar = min(10, N_CB)
methods_cmp = [
    ('FAISS+Smooth', smooth_codes[0, :, :L_v].numpy()),
    ('V2 (t=0.8)', v2_results[0.8][0][0, :, :L_v].numpy()),
    ('Hybrid', hybrid_codes[0, :, :L_v].numpy()),
]
src_ref = src_codes[0, :, :L_v].numpy()
x = np.arange(n_show_bar)
w = 0.25
for i, (name, c) in enumerate(methods_cmp):
    rates = [np.mean(c[cb] != src_ref[cb]) * 100 for cb in range(n_show_bar)]
    ax.bar(x + i*w - w, rates, w, label=name, alpha=0.8)
ax.set_xlabel('Codebook'); ax.set_ylabel('% Changed from Source')
ax.set_xticks(range(n_show_bar))
ax.set_xticklabels([f'CB{i}' for i in range(n_show_bar)])
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
ax.set_title(f'Token Change Rate vs Source (first {n_show_bar} CBs)')
plt.savefig('/content/change_rate.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 3. Mel spectrogram comparison ---
fig, axes = plt.subplots(5, 1, figsize=(14, 12), constrained_layout=True)
spec_items = [
    ('Source', source_wav),
    ('V2 temp=0.8', v2_results[0.8][1]),
    ('Hybrid (V2+Ret)', hybrid_wav),
    ('FAISS+Smooth', smooth_wav),
    ('Target ref', target_wav_concat[:int(SR*15)]),
]
for ax, (lbl, w) in zip(axes, spec_items):
    mel = librosa.feature.melspectrogram(y=w, sr=SR, n_mels=80, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=SR, x_axis='time', y_axis='mel', ax=ax, fmax=8000)
    ax.set_title(lbl, fontsize=10)
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Mel Spectrograms — Transformer vs Retrieval', fontsize=13)
plt.savefig('/content/spectrograms_transformer.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 4. CB1 token distribution histogram ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, (lbl, tokens) in zip(axes, [
    ('Target GT CB1', all_tgt_codes[0, 1, :L_v].numpy()),
    ('V2 pred CB1 (t=0.8)', v2_results[0.8][2][0, :L_v].numpy()),
    ('FAISS+Smooth CB1', smooth_codes[0, 1, :L_v].numpy()),
]):
    c = Counter(tokens.tolist())
    vals, counts = zip(*c.most_common(30))
    ax.bar(range(len(vals)), counts, color='#4472C4', alpha=0.8)
    ax.set_title(f'{lbl} (unique={len(c)})', fontsize=10)
    ax.set_xlabel('Token rank'); ax.set_ylabel('Count')
plt.suptitle('CB1 Token Distribution (top-30)', fontsize=13)
plt.savefig('/content/cb1_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 5. Confidence heatmap (first N_DIAG CBs) ---
print('\nPrediction confidence heatmap (V2 on source):')
with torch.no_grad():
    v2_logits_src = model_v2(src_sem_d, src_acou_d, tgt_acou_tokens=None)

n_conf = min(N_DIAG, N_ACOU)
conf_len = min(T_src, 200)
conf_map = np.zeros((n_conf, conf_len))
for cb_i in range(n_conf):
    probs = F.softmax(v2_logits_src[0, cb_i, :conf_len].cpu(), dim=-1)
    conf_map[cb_i] = probs.max(dim=-1).values.numpy()

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(conf_map, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1, interpolation='nearest')
ax.set_ylabel('CB'); ax.set_xlabel('Frame')
ax.set_yticks(range(n_conf)); ax.set_yticklabels([f'CB{i+1}' for i in range(n_conf)])
ax.set_title(f'V2 Prediction Confidence — first {n_conf} CBs (green=high, red=low)')
plt.colorbar(im, ax=ax, shrink=0.5)
plt.savefig('/content/confidence_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean confidence per CB: {[f"CB{i+1}={conf_map[i].mean():.3f}" for i in range(n_conf)]}')

In [ ]:
# ==============================================================
# CELL 8: Playback all results (Retrieval + Transformer)
# ==============================================================

print('='*70)
print('PLAYBACK — Listen to all methods')
print('='*70)

results = [
    ('0. Source (original)', source_wav),
    ('1. Naive Swap', naive_wav),
    ('2. Lookup Retrieval', lookup_wav),
    ('3. FAISS Retrieval', faiss_wav),
    ('4. FAISS+Smooth', smooth_wav),
    ('5. FAISS+Smooth keep CB0', smooth_keepcb0_wav),
    ('6. Transformer V2 (temp=0.8)', v2_results[0.8][1]),
    ('7. Hybrid (V2 CB1-3 + Retrieval CB4-7)', hybrid_wav),
    ('T. Target speaker reference', target_wav_concat[:int(SR*15)]),
]

for label, wav in results:
    print(f'\n{label}:')
    display(PlayAudio(wav, rate=SR))

In [ ]:
# ==============================================================
# CELL 9: Detailed diagnostic — token-level analysis
# ==============================================================

print('='*70)
print('DETAILED TOKEN ANALYSIS')
print('='*70)

L = min(T_src, 200)  # analyze first 200 frames
N_SHOW = min(N_CB, 10)  # show first 10 CBs in tables for readability

all_code_sets = [
    ('Source', src_codes),
    ('Naive Swap', naive_codes),
    ('Lookup', lookup_codes),
    ('FAISS', faiss_codes),
    ('FAISS+Smooth', smooth_codes),
    ('FAISS+Smooth keepCB0', smooth_keepcb0_codes),
    ('V2 (t=0.8)', v2_results[0.8][0]),
    ('Hybrid', hybrid_codes),
]

tgt_ref = all_tgt_codes
L_tgt = min(T_tgt, L)

# Per-codebook change analysis
print(f'\n--- Per-codebook token change rate (vs source) [first {N_SHOW} CBs] ---')
print(f'{"Method":<25}', end='')
for cb in range(N_SHOW):
    print(f' CB{cb:>2}', end='')
print()
print('-' * (25 + N_SHOW * 5))

src_t = src_codes[0, :, :L].numpy()
for name, codes in all_code_sets:
    c = codes[0, :, :L].numpy()
    print(f'{name:<25}', end='')
    for cb in range(N_SHOW):
        rate = np.mean(c[cb] != src_t[cb]) * 100
        print(f' {rate:3.0f}%', end='')
    print()

# Per-codebook match rate with target
print(f'\n--- Per-codebook token match rate (vs target) [first {N_SHOW} CBs] ---')
print(f'{"Method":<25}', end='')
for cb in range(N_SHOW):
    print(f' CB{cb:>2}', end='')
print()
print('-' * (25 + N_SHOW * 5))

tgt_t = tgt_ref[0, :, :L].numpy()
for name, codes in all_code_sets:
    c = codes[0, :, :min(L, codes.shape[2])].numpy()
    cL = c.shape[1]
    print(f'{name:<25}', end='')
    for cb in range(N_SHOW):
        rate = np.mean(c[cb, :min(cL, L_tgt)] == tgt_t[cb, :min(cL, L_tgt)]) * 100
        print(f' {rate:3.0f}%', end='')
    print()

# Codebook distribution analysis
print(f'\n--- CB1 token distribution (most important acoustic CB) ---')
src_cb1 = Counter(src_codes[0, 1, :L].numpy().tolist())
tgt_cb1 = Counter(all_tgt_codes[0, 1, :L_tgt].numpy().tolist())
smooth_cb1 = Counter(smooth_codes[0, 1, :L].numpy().tolist())
v2_cb1 = Counter(v2_results[0.8][0][0, 1, :L].numpy().tolist())

print(f'  Source CB1:  {len(src_cb1):4d} unique tokens, top-5: {src_cb1.most_common(5)}')
print(f'  Target CB1:  {len(tgt_cb1):4d} unique tokens, top-5: {tgt_cb1.most_common(5)}')
print(f'  Smooth CB1:  {len(smooth_cb1):4d} unique tokens, top-5: {smooth_cb1.most_common(5)}')
print(f'  V2 CB1:      {len(v2_cb1):4d} unique tokens, top-5: {v2_cb1.most_common(5)}')

# Retrieval quality per frame
print(f'\n--- Retrieval quality for FAISS+Smooth ---')
sel_sims = dists_k[np.arange(T_src), 0]
print(f'  Top-1 cosine similarity:')
print(f'    Mean:   {sel_sims.mean():.4f}')
print(f'    Std:    {sel_sims.std():.4f}')
print(f'    Min:    {sel_sims.min():.4f}')
print(f'    Max:    {sel_sims.max():.4f}')
print(f'    < 0.5:  {np.sum(sel_sims < 0.5)}/{T_src} frames ({100*np.sum(sel_sims < 0.5)/T_src:.1f}%) [poor]')
print(f'    < 0.7:  {np.sum(sel_sims < 0.7)}/{T_src} frames ({100*np.sum(sel_sims < 0.7)/T_src:.1f}%)')
print(f'    > 0.9:  {np.sum(sel_sims > 0.9)}/{T_src} frames ({100*np.sum(sel_sims > 0.9)/T_src:.1f}%) [good]')
print(f'    > 0.95: {np.sum(sel_sims > 0.95)}/{T_src} frames ({100*np.sum(sel_sims > 0.95)/T_src:.1f}%) [excellent]')

In [ ]:
# ==============================================================
# CELL 10: Visual diagnostics
# ==============================================================

print('='*70)
print('VISUAL DIAGNOSTICS')
print('='*70)

L_vis = min(T_src, T_tgt, 200)

# --- 1. Token heatmaps ---
fig, axes = plt.subplots(6, 1, figsize=(16, 16), constrained_layout=True)

vis_items = [
    ('Source', src_codes[0, :, :L_vis]),
    ('Target Reference', all_tgt_codes[0, :, :L_vis]),
    ('Naive Swap', naive_codes[0, :, :L_vis]),
    ('FAISS Retrieval', faiss_codes[0, :, :L_vis]),
    ('FAISS+Smooth', smooth_codes[0, :, :L_vis]),
    ('FAISS+Smooth (keep CB0)', smooth_keepcb0_codes[0, :, :L_vis]),
]

for ax, (title, codes) in zip(axes, vis_items):
    im = ax.imshow(codes.numpy(), aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_ylabel('CB')
    ax.set_title(title, fontsize=10)
    ax.set_yticks(range(0, N_CB, max(1, N_CB//8)))
    plt.colorbar(im, ax=ax, shrink=0.5)

axes[-1].set_xlabel('Frame (12.5 Hz)')
plt.suptitle('Token Heatmaps — All Methods', fontsize=13)
plt.savefig('/content/heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 2. Retrieval similarity histogram ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)

axes[0].hist(dists_k[:, 0], bins=50, color='#4472C4', edgecolor='white')
axes[0].axvline(dists_k[:, 0].mean(), color='red', linestyle='--', label=f'mean={dists_k[:, 0].mean():.3f}')
axes[0].set_xlabel('Cosine Similarity (top-1)')
axes[0].set_ylabel('Count')
axes[0].set_title('FAISS Retrieval Quality per Frame')
axes[0].legend()

axes[1].plot(np.abs(np.diff(selected)), color='#C55A11', linewidth=0.5)
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Index Jump')
axes[1].set_title(f'Temporal Continuity (smooth={np.sum(jumps<=5)/len(jumps)*100:.0f}% jumps<=5)')
axes[1].set_ylim(0, min(jumps.max(), 500))

plt.savefig('/content/retrieval_quality.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 3. Waveforms ---
fig, axes = plt.subplots(4, 1, figsize=(14, 8), constrained_layout=True)
for ax, (lbl, w), clr in zip(axes,
    [('Source', source_wav), ('FAISS+Smooth', smooth_wav),
     ('FAISS+Smooth (keep CB0)', smooth_keepcb0_wav),
     ('Target ref', target_wav_concat[:int(SR*15)])],
    ['#4472C4', '#C55A11', '#7030A0', '#548235']):
    t_ax = np.arange(len(w)) / SR
    ax.plot(t_ax, w, color=clr, linewidth=0.3)
    ax.set_title(lbl, fontsize=10)
    ax.set_ylabel('Amp')
    ax.set_xlim(0, t_ax[-1])
    ax.grid(True, alpha=0.2)
axes[-1].set_xlabel('Time (s)')
plt.savefig('/content/waveforms.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 4. Mel spectrograms ---
fig, axes = plt.subplots(4, 1, figsize=(14, 8), constrained_layout=True)
for ax, (lbl, w) in zip(axes,
    [('Source', source_wav), ('FAISS+Smooth', smooth_wav),
     ('FAISS+Smooth (keep CB0)', smooth_keepcb0_wav),
     ('Target ref', target_wav_concat[:int(SR*15)])]):
    mel = librosa.feature.melspectrogram(y=w, sr=SR, n_mels=80, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=SR, x_axis='time', y_axis='mel', ax=ax, fmax=8000)
    ax.set_title(lbl, fontsize=10)
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Mel Spectrograms', fontsize=13)
plt.savefig('/content/spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ==============================================================
# CELL 11: Objective metrics (all methods incl. Transformer)
# ==============================================================

print('='*70)
print('OBJECTIVE METRICS')
print('='*70)

from resemblyzer import VoiceEncoder, preprocess_wav
try: spk_enc
except: spk_enc = VoiceEncoder(device=device)

def compute_secs(a, b):
    a16 = librosa.resample(a.astype(np.float32), orig_sr=SR, target_sr=16000)
    b16 = librosa.resample(b.astype(np.float32), orig_sr=SR, target_sr=16000)
    ea = spk_enc.embed_utterance(a16)
    eb = spk_enc.embed_utterance(b16)
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

def compute_mcd(ref, syn):
    r = librosa.feature.mfcc(y=ref.astype(np.float32), sr=SR, n_mfcc=13)
    s = librosa.feature.mfcc(y=syn.astype(np.float32), sr=SR, n_mfcc=13)
    L = min(r.shape[1], s.shape[1])
    if L == 0: return None
    diff = r[1:, :L] - s[1:, :L]
    return float(np.mean(np.sqrt(2.0 * np.sum(diff**2, axis=0))))

tgt_ref_wav = target_wav_concat[:int(SR*15)]

src_src_secs = compute_secs(source_wav, source_wav)
src_tgt_secs = compute_secs(source_wav, tgt_ref_wav)
tgt_tgt_secs = compute_secs(tgt_ref_wav, tgt_ref_wav)

print(f'\n--- Speaker Embedding Baselines ---')
print(f'  Source vs Source:  {src_src_secs:.4f} (self-similarity)')
print(f'  Target vs Target:  {tgt_tgt_secs:.4f} (self-similarity)')
print(f'  Source vs Target:  {src_tgt_secs:.4f} (cross-speaker)')

print(f'\n--- All Conversion Results ---')
print(f'{"Method":<30} {"SECS(->tgt)":>12} {"SECS(->src)":>12} {"MCD":>8}')
print('-'*65)

eval_methods = [
    ('Naive Swap', naive_wav),
    ('Lookup Retrieval', lookup_wav),
    ('FAISS Retrieval', faiss_wav),
    ('FAISS+Smooth', smooth_wav),
    ('FAISS+Smooth (keep CB0)', smooth_keepcb0_wav),
    ('Transformer V2 (t=0.2)', v2_results[0.2][1]),
    ('Transformer V2 (t=0.5)', v2_results[0.5][1]),
    ('Transformer V2 (t=0.8)', v2_results[0.8][1]),
    ('Transformer V2 (t=1.0)', v2_results[1.0][1]),
    ('Hybrid (V2+Ret)', hybrid_wav),
]

for name, wav in eval_methods:
    secs_tgt = compute_secs(wav, tgt_ref_wav)
    secs_src = compute_secs(wav, source_wav)
    mcd = compute_mcd(source_wav, wav)
    print(f'{name:<30} {secs_tgt:>12.4f} {secs_src:>12.4f} {mcd:>8.2f}')

print(f'\n  SECS(->tgt): higher = sounds more like target speaker')
print(f'  SECS(->src): higher = still sounds like source (less conversion)')
print(f'  MCD: lower = less spectral distortion')

In [ ]:
# ==============================================================
# CELL 12: Save all results & print summary
# ==============================================================

print('='*70)
print('SAVING RESULTS')
print('='*70)

save_items = [
    ('naive_swap.wav', naive_wav),
    ('lookup_retrieval.wav', lookup_wav),
    ('faiss_retrieval.wav', faiss_wav),
    ('faiss_smooth.wav', smooth_wav),
    ('faiss_smooth_keepcb0.wav', smooth_keepcb0_wav),
    ('transformer_v2_t02.wav', v2_results[0.2][1]),
    ('transformer_v2_t05.wav', v2_results[0.5][1]),
    ('transformer_v2_t08.wav', v2_results[0.8][1]),
    ('transformer_v2_t10.wav', v2_results[1.0][1]),
    ('hybrid_v2_ret.wav', hybrid_wav),
]

for fname, wav in save_items:
    sf.write(f'/content/{fname}', wav, SR)
    print(f'  Saved: /content/{fname}')

print(f'\n{"="*70}')
print('SUMMARY')
print(f'{"="*70}')
print(f'Target speaker: {total_target_dur:.1f}s, {T_tgt} frames, {len(lookup)} unique CB0 tokens')
print(f'Source audio:   {source_dur:.1f}s, {T_src} frames, {len(set(src_cb0.tolist()))} unique CB0 tokens')
print(f'CB0 coverage:   {100*n_matched/T_src:.1f}% exact match')
print(f'FAISS quality:  avg top-1 sim = {dists_k[:, 0].mean():.4f}')
print(f'Smoothing:      {np.sum(jumps<=5)/len(jumps)*100:.0f}% smooth transitions')
print(f'Transformer V2: {n_params/1e6:.1f}M params, trained {train_time:.0f}s, final loss={all_losses[-1]:.4f}')
print(f'  CB1 acc (TF): {per_cb_accs[0][-1]:.1f}%')

print(f'\nCopy the full output of this notebook and share for analysis.')

In [ ]:
# ==============================================================
# CELL 13: Download all converted audio
# ==============================================================

!cd /content && zip -q converted_results.zip *.wav *.png
files.download('/content/converted_results.zip')